# 02 - Baselines (Thea)

Models **1-3** of the results table: majority class, logistic regression and random forest on hand-made colour features.

Rules: train on `train`, **choose every setting on `val`**, never look at `test` until the very end. Outputs: `predictions/<model>_<split>_logits.npy` (float32, `(N, 56)`, rows in `manifest.csv` order, columns in `classes.json` order).

No GPU needed.

In [ ]:
import json, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from geosense import load_classes, load_manifest
from geosense.data import PROJECT_ROOT
from geosense.predictions import PRED_DIR, check_val, save_logits
from geosense.preprocess import find_dataset_root

# --- configuration ---------------------------------------------------------------------------
DATA_ROOT = find_dataset_root()      # folder that contains France/, Japan/, ...  (or pass the path)
SUBSET = None                        # dry run: e.g. 1 = one image per country and split. None = everything
# ---------------------------------------------------------------------------------------------
classes, class_weights = load_classes()
C = len(classes)
manifest = load_manifest()
if SUBSET:                           # quick test of the whole notebook; writes to predictions_dryrun/
    manifest = manifest.groupby(["split", "country"], group_keys=False).head(SUBSET).reset_index(drop=True)
OUT = PRED_DIR if SUBSET is None else PROJECT_ROOT / "predictions_dryrun"
part = {s: manifest[manifest.split == s].reset_index(drop=True) for s in ("train", "val", "test")}
y = {s: part[s].label.to_numpy() for s in part}
print(DATA_ROOT, "|", C, "classes |", {s: len(d) for s, d in part.items()}, "| output:", OUT)

## 1. Majority class
Always guesses the most frequent training country (USA). Logit = log of the training class frequency.

In [ ]:
from geosense.baselines import majority_logits, macro_f1
for s in ('val', 'test'):
    save_logits('majority', s, majority_logits(y['train'], C, len(part[s])), OUT, manifest, C)
lg = majority_logits(y['train'], C, len(part['val']))
print('val accuracy', (lg.argmax(1) == y['val']).mean().round(3), '| val macro-F1', round(macro_f1(y['val'], lg), 3))

## 2. Hand-made features
Colour histograms (sky and ground separately), mean/std, edge energy, hue, green and blue share. Computed on the **cropped** image (interface removed). About 5-10 min on 4 CPU cores for all images; cached in `features/`.

In [ ]:
from geosense.baselines import colour_features
(PROJECT_ROOT / 'features').mkdir(exist_ok=True)
X = {}
for s in ('train', 'val', 'test'):
    f = PROJECT_ROOT / 'features' / f'colour_{s}{"" if SUBSET is None else "_dry"}.npy'
    if f.exists():
        X[s] = np.load(f)
    else:
        t = time.time(); X[s] = colour_features(DATA_ROOT, part[s].path); np.save(f, X[s]); print(s, f'{time.time()-t:.0f}s')
print({s: v.shape for s, v in X.items()})

## 3. Logistic regression (settings chosen on val by macro-F1)
`class_weight='balanced'` handles the imbalance (USA = 25% of the data).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from geosense.predictions import full_class_columns, probs_to_logits

def to_logits(model, Xs):
    return probs_to_logits(full_class_columns(model.predict_proba(Xs), model.classes_, C))

best = (-1, None, None)
for Creg in (0.01, 0.1, 1.0, 10.0):
    m = make_pipeline(StandardScaler(), LogisticRegression(C=Creg, class_weight='balanced', max_iter=500)).fit(X['train'], y['train'])
    f1 = macro_f1(y['val'], to_logits(m, X['val']))
    print(f'C={Creg:<5} val macro-F1 {f1:.3f}')
    if f1 > best[0]: best = (f1, Creg, m)
lr_f1, lr_C, lr = best
print('chosen C =', lr_C)
for s in ('val', 'test'):
    save_logits('histogram_lr', s, to_logits(lr, X[s]), OUT, manifest, C)

## 4. Random forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
best = (-1, None, None)
for leaf in (1, 3):
    for depth in (None, 25):
        m = RandomForestClassifier(n_estimators=300, min_samples_leaf=leaf, max_depth=depth, max_features='sqrt',
                                   class_weight='balanced_subsample', n_jobs=-1, random_state=42).fit(X['train'], y['train'])
        f1 = macro_f1(y['val'], to_logits(m, X['val']))
        print(f'leaf={leaf} depth={depth}: val macro-F1 {f1:.3f}')
        if f1 > best[0]: best = (f1, (leaf, depth), m)
rf_f1, rf_params, rf = best
print('chosen (leaf, depth) =', rf_params)
for s in ('val', 'test'):
    save_logits('histogram_rf', s, to_logits(rf, X[s]), OUT, manifest, C)

## 5. Check the exported files (validation only)

In [ ]:
check_val(OUT, manifest)

### Notes for the report
- Why these baselines: the floor (25% accuracy but macro-F1 near 0) and what colour alone can do (Seif's EDA: colour is ~2x chance, a weak real signal).
- Class weights: `balanced` (LR) / `balanced_subsample` (RF); settings chosen on **val macro-F1**.
- Push `predictions/*.npy` to GitHub (small files) and tell Maya.